<a href="https://colab.research.google.com/github/AshokGit544/Enterprise-RAGAS-Evaluation-Platform/blob/main/Enterprise_Ragas_Evaluation_Platform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install pandas numpy scikit-learn

In [2]:
from pathlib import Path
from datetime import datetime
import random
import json
import re

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [3]:
BASE_PATH = Path('/content/enterprise_ragas_evaluation_platform')
RAW_PATH = BASE_PATH / 'data' / 'raw_documents'
PROCESSED_PATH = BASE_PATH / 'data' / 'processed_documents'
CHUNK_PATH = BASE_PATH / 'data' / 'chunked_documents'
VECTOR_PATH = BASE_PATH / 'data' / 'vector_ready'
OUTPUT_PATH = BASE_PATH / 'outputs'
CONFIG_PATH = BASE_PATH / 'config'

for p in [RAW_PATH, PROCESSED_PATH, CHUNK_PATH, VECTOR_PATH, OUTPUT_PATH, CONFIG_PATH]:
    p.mkdir(parents=True, exist_ok=True)

random.seed(42)
np.random.seed(42)

print('Project folders created successfully.')
print(BASE_PATH)

Project folders created successfully.
/content/enterprise_ragas_evaluation_platform


In [4]:
project_schema = {
    'document_types': [
        'quality_report',
        'supplier_invoice',
        'maintenance_record',
        'engineering_change_notice'
    ],
    'evaluation_focus': [
        'retrieval_quality',
        'groundedness',
        'answer_relevance',
        'context_precision',
        'context_recall'
    ],
    'project_goal': 'evaluate_rag_quality_for_enterprise_document_intelligence'
}

with open(CONFIG_PATH / 'project_schema.json', 'w') as f:
    json.dump(project_schema, f, indent=2)

print(json.dumps(project_schema, indent=2))

{
  "document_types": [
    "quality_report",
    "supplier_invoice",
    "maintenance_record",
    "engineering_change_notice"
  ],
  "evaluation_focus": [
    "retrieval_quality",
    "groundedness",
    "answer_relevance",
    "context_precision",
    "context_recall"
  ],
  "project_goal": "evaluate_rag_quality_for_enterprise_document_intelligence"
}


In [5]:
documents = []

supplier_names = ['Magna', 'Bosch', 'Denso', 'Lear', 'Aptiv']
plants = ['PLT100', 'PLT200', 'PLT300', 'PLT400']
issue_types = ['weld defect', 'paint inconsistency', 'sensor failure', 'material shortage', 'assembly variance']
equipment_ids = ['EQP1001', 'EQP1002', 'EQP1003', 'EQP1004']

for i in range(1, 401):
    doc_type = random.choice(project_schema['document_types'])
    plant = random.choice(plants)
    supplier = random.choice(supplier_names)
    issue = random.choice(issue_types)
    equipment = random.choice(equipment_ids)
    doc_date = f"2024-{random.randint(1,12):02d}-{random.randint(1,28):02d}"
    amount = max(1000.0, round(float(np.random.normal(12500, 3500)), 2))
    change_order_id = f"ECO-{random.randint(10000,99999)}"

    if doc_type == 'quality_report':
        text = (
            f"Document ID DOC{i:05d}. Plant {plant}. Quality inspection found {issue} on equipment {equipment}. "
            f"Document date {doc_date}. Supplier reference {supplier}. "
            f"The report explains defect review, containment action, severity impact, and corrective follow-up."
        )
    elif doc_type == 'supplier_invoice':
        text = (
            f"Document ID DOC{i:05d}. Supplier invoice from {supplier} for plant {plant}. "
            f"Invoice amount ${amount}. Document date {doc_date}. Material issue related to {issue}. "
            f"The invoice includes purchasing context, billing dependency, and cost review notes."
        )
    elif doc_type == 'maintenance_record':
        text = (
            f"Document ID DOC{i:05d}. Maintenance record for equipment {equipment} at plant {plant}. "
            f"Reported issue {issue}. Service date {doc_date}. Vendor {supplier}. "
            f"The note explains service action, downtime impact, and equipment recovery details."
        )
    else:
        text = (
            f"Document ID DOC{i:05d}. Engineering change notice {change_order_id} for plant {plant}. "
            f"Change related to {issue}. Effective date {doc_date}. Supplier {supplier}. "
            f"The notice explains implementation details, downstream manufacturing effect, and engineering review."
        )

    documents.append({
        'document_id': f'DOC{i:05d}',
        'document_type': doc_type,
        'document_text': text
    })

documents_df = pd.DataFrame(documents)
documents_df.to_csv(RAW_PATH / 'enterprise_documents.csv', index=False)

print(documents_df.head())
print('Total documents:', len(documents_df))

  document_id   document_type  \
0    DOC00001  quality_report   
1    DOC00002  quality_report   
2    DOC00003  quality_report   
3    DOC00004  quality_report   
4    DOC00005  quality_report   

                                       document_text  
0  Document ID DOC00001. Plant PLT100. Quality in...  
1  Document ID DOC00002. Plant PLT400. Quality in...  
2  Document ID DOC00003. Plant PLT200. Quality in...  
3  Document ID DOC00004. Plant PLT200. Quality in...  
4  Document ID DOC00005. Plant PLT100. Quality in...  
Total documents: 400


In [6]:
def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def extract_field(pattern, text, group_index=1):
    match = re.search(pattern, text, re.IGNORECASE)
    if match:
        return match.group(group_index)
    return None

documents_df = pd.read_csv(RAW_PATH / 'enterprise_documents.csv')
documents_df['normalized_text'] = documents_df['document_text'].apply(normalize_text)
documents_df['plant_code'] = documents_df['document_text'].apply(lambda x: extract_field(r'plant\s+(PLT\d+)', x))
documents_df['supplier_name'] = documents_df['document_text'].apply(
    lambda x: extract_field(r'(?:supplier reference|supplier invoice from|vendor|supplier)\s+(Magna|Bosch|Denso|Lear|Aptiv)', x)
)
documents_df['equipment_id'] = documents_df['document_text'].apply(lambda x: extract_field(r'equipment\s+(EQP\d+)', x))
documents_df['document_date'] = documents_df['document_text'].apply(lambda x: extract_field(r'(\d{4}-\d{2}-\d{2})', x))
documents_df['change_order_id'] = documents_df['document_text'].apply(lambda x: extract_field(r'(ECO-\d+)', x))

documents_df.to_csv(PROCESSED_PATH / 'processed_documents.csv', index=False)
print(documents_df.head())

  document_id   document_type  \
0    DOC00001  quality_report   
1    DOC00002  quality_report   
2    DOC00003  quality_report   
3    DOC00004  quality_report   
4    DOC00005  quality_report   

                                       document_text  \
0  Document ID DOC00001. Plant PLT100. Quality in...   
1  Document ID DOC00002. Plant PLT400. Quality in...   
2  Document ID DOC00003. Plant PLT200. Quality in...   
3  Document ID DOC00004. Plant PLT200. Quality in...   
4  Document ID DOC00005. Plant PLT100. Quality in...   

                                     normalized_text plant_code supplier_name  \
0  document id doc00001. plant plt100. quality in...     PLT100         Denso   
1  document id doc00002. plant plt400. quality in...     PLT400         Magna   
2  document id doc00003. plant plt200. quality in...     PLT200         Aptiv   
3  document id doc00004. plant plt200. quality in...     PLT200          Lear   
4  document id doc00005. plant plt100. quality in...     PL

In [7]:
def simple_chunk_text(text, chunk_size=28):
    words = str(text).split()
    chunks = []
    for i in range(0, len(words), chunk_size):
        chunk = ' '.join(words[i:i + chunk_size])
        if chunk.strip():
            chunks.append(chunk)
    return chunks

chunk_rows = []

for _, row in documents_df.iterrows():
    chunks = simple_chunk_text(row['normalized_text'], chunk_size=28)

    for idx, chunk in enumerate(chunks, start=1):
        chunk_rows.append({
            'document_id': row['document_id'],
            'chunk_id': f"{row['document_id']}_CHUNK_{idx}",
            'document_type': row['document_type'],
            'plant_code': row['plant_code'],
            'supplier_name': row['supplier_name'],
            'equipment_id': row['equipment_id'],
            'document_date': row['document_date'],
            'change_order_id': row['change_order_id'],
            'chunk_text': chunk
        })

chunk_df = pd.DataFrame(chunk_rows)
chunk_df.to_csv(CHUNK_PATH / 'chunked_documents.csv', index=False)

print(chunk_df.head())
print('Total chunks:', len(chunk_df))

  document_id          chunk_id   document_type plant_code supplier_name  \
0    DOC00001  DOC00001_CHUNK_1  quality_report     PLT100         Denso   
1    DOC00001  DOC00001_CHUNK_2  quality_report     PLT100         Denso   
2    DOC00002  DOC00002_CHUNK_1  quality_report     PLT400         Magna   
3    DOC00002  DOC00002_CHUNK_2  quality_report     PLT400         Magna   
4    DOC00003  DOC00003_CHUNK_1  quality_report     PLT200         Aptiv   

  equipment_id document_date change_order_id  \
0      EQP1002    2024-03-24            None   
1      EQP1002    2024-03-24            None   
2      EQP1001    2024-04-08            None   
3      EQP1001    2024-04-08            None   
4      EQP1002    2024-08-19            None   

                                          chunk_text  
0  document id doc00001. plant plt100. quality in...  
1                          and corrective follow-up.  
2  document id doc00002. plant plt400. quality in...  
3                          and cor

In [8]:
chunk_df['vector_ready_text'] = (
    'DocumentType: ' + chunk_df['document_type'].fillna('UNKNOWN').astype(str) + ' | ' +
    'Plant: ' + chunk_df['plant_code'].fillna('UNKNOWN').astype(str) + ' | ' +
    'Supplier: ' + chunk_df['supplier_name'].fillna('UNKNOWN').astype(str) + ' | ' +
    'Equipment: ' + chunk_df['equipment_id'].fillna('UNKNOWN').astype(str) + ' | ' +
    'Date: ' + chunk_df['document_date'].fillna('UNKNOWN').astype(str) + ' | ' +
    'ChangeOrder: ' + chunk_df['change_order_id'].fillna('UNKNOWN').astype(str) + ' | ' +
    'Text: ' + chunk_df['chunk_text'].fillna('').astype(str)
)

chunk_df.to_csv(VECTOR_PATH / 'vector_ready_records.csv', index=False)

print(chunk_df[['chunk_id', 'vector_ready_text']].head())

           chunk_id                                  vector_ready_text
0  DOC00001_CHUNK_1  DocumentType: quality_report | Plant: PLT100 |...
1  DOC00001_CHUNK_2  DocumentType: quality_report | Plant: PLT100 |...
2  DOC00002_CHUNK_1  DocumentType: quality_report | Plant: PLT400 |...
3  DOC00002_CHUNK_2  DocumentType: quality_report | Plant: PLT400 |...
4  DOC00003_CHUNK_1  DocumentType: quality_report | Plant: PLT200 |...


In [9]:
retrieval_vectorizer = TfidfVectorizer(stop_words='english')
retrieval_matrix = retrieval_vectorizer.fit_transform(chunk_df['vector_ready_text'])

print('Retrieval matrix shape:', retrieval_matrix.shape)

Retrieval matrix shape: (800, 757)


In [10]:
evaluation_queries = [
    {
        'query_id': 'Q1',
        'question': 'What quality issue was found for equipment EQP1002 in plant PLT200?',
        'expected_keywords': ['quality', 'issue', 'equipment', 'eqp1002', 'plant', 'plt200']
    },
    {
        'query_id': 'Q2',
        'question': 'Which supplier invoice mentions material issue and cost review?',
        'expected_keywords': ['supplier', 'invoice', 'material', 'issue', 'cost', 'review']
    },
    {
        'query_id': 'Q3',
        'question': 'What maintenance record mentions downtime impact and equipment recovery?',
        'expected_keywords': ['maintenance', 'downtime', 'equipment', 'recovery']
    },
    {
        'query_id': 'Q4',
        'question': 'Which engineering change notice explains downstream manufacturing effect?',
        'expected_keywords': ['engineering', 'change', 'downstream', 'manufacturing', 'effect']
    }
]

evaluation_df = pd.DataFrame(evaluation_queries)
evaluation_df.to_csv(CONFIG_PATH / 'evaluation_queries.csv', index=False)

evaluation_df

,query_id,question,expected_keywords
0,Q1,What quality issue was found for equipment EQP...,"[quality, issue, equipment, eqp1002, plant, pl..."
1,Q2,Which supplier invoice mentions material issue...,"[supplier, invoice, material, issue, cost, rev..."
2,Q3,What maintenance record mentions downtime impa...,"[maintenance, downtime, equipment, recovery]"
3,Q4,Which engineering change notice explains downs...,"[engineering, change, downstream, manufacturin..."


In [11]:
def retrieve_top_k_chunks(question, top_k=5):
    query_vector = retrieval_vectorizer.transform([question])
    similarity_scores = cosine_similarity(query_vector, retrieval_matrix).flatten()
    top_indices = similarity_scores.argsort()[-top_k:][::-1]

    results_df = chunk_df.iloc[top_indices].copy()
    results_df['similarity_score'] = similarity_scores[top_indices]
    results_df = results_df.reset_index(drop=True)
    return results_df

In [12]:
def first_non_null(series, default='UNKNOWN'):
    valid = series.dropna()
    if len(valid) > 0:
        return str(valid.iloc[0])
    return default

def generate_grounded_response(question, retrieved_df):
    top_rows = retrieved_df.head(3)

    document_type = first_non_null(top_rows['document_type'])
    plant_code = first_non_null(top_rows['plant_code'])
    equipment_id = first_non_null(top_rows['equipment_id'])
    supplier_name = first_non_null(top_rows['supplier_name'])

    detected_issue = 'not clearly identified'
    issue_pattern = r'(weld defect|paint inconsistency|sensor failure|material shortage|assembly variance)'

    for text in top_rows['chunk_text'].astype(str):
        match = re.search(issue_pattern, text, re.IGNORECASE)
        if match:
            detected_issue = match.group(1).lower()
            break

    evidence_snippets = []
    for _, row in top_rows.iterrows():
        evidence_snippets.append(
            f"{row['document_id']} | {row['document_type']} | {row['chunk_text']}"
        )

    response = (
        f"Based on the retrieved enterprise context, the most relevant document type is {document_type}. "
        f"The response points to plant {plant_code}, equipment {equipment_id}, supplier {supplier_name}, "
        f"and the main issue appears to be {detected_issue}. "
        f"This answer is grounded in retrieved enterprise document chunks rather than model-only memory. "
        f"Supporting evidence: {' || '.join(evidence_snippets)}"
    )

    return response

In [13]:
def tokenize_text(text):
    text = str(text).lower()
    return set(re.findall(r'\b[a-z0-9\-]+\b', text))

def compute_answer_relevance(question, answer):
    q_tokens = tokenize_text(question)
    a_tokens = tokenize_text(answer)
    if len(q_tokens) == 0:
        return 0.0
    return round(len(q_tokens.intersection(a_tokens)) / len(q_tokens), 4)

def compute_groundedness(answer, contexts):
    answer_tokens = tokenize_text(answer)
    context_tokens = set()

    for ctx in contexts:
        context_tokens.update(tokenize_text(ctx))

    if len(answer_tokens) == 0:
        return 0.0
    return round(len(answer_tokens.intersection(context_tokens)) / len(answer_tokens), 4)

def compute_context_precision(expected_keywords, contexts):
    context_tokens = set()
    for ctx in contexts:
        context_tokens.update(tokenize_text(ctx))

    expected_tokens = set([kw.lower() for kw in expected_keywords])

    if len(context_tokens) == 0:
        return 0.0
    return round(len(expected_tokens.intersection(context_tokens)) / len(expected_tokens), 4)

def compute_context_recall(expected_keywords, contexts):
    context_tokens = set()
    for ctx in contexts:
        context_tokens.update(tokenize_text(ctx))

    expected_tokens = set([kw.lower() for kw in expected_keywords])

    if len(expected_tokens) == 0:
        return 0.0
    return round(len(expected_tokens.intersection(context_tokens)) / len(expected_tokens), 4)

In [14]:
evaluation_results = []

for _, row in evaluation_df.iterrows():
    question = row['question']
    expected_keywords = row['expected_keywords']

    if isinstance(expected_keywords, str):
        # safety fallback if loaded from csv later
        expected_keywords = re.findall(r"[A-Za-z0-9\-_]+", expected_keywords)

    retrieved_df = retrieve_top_k_chunks(question, top_k=5)
    contexts = retrieved_df['chunk_text'].astype(str).tolist()
    answer = generate_grounded_response(question, retrieved_df)

    answer_relevance = compute_answer_relevance(question, answer)
    groundedness = compute_groundedness(answer, contexts)
    context_precision = compute_context_precision(expected_keywords, contexts)
    context_recall = compute_context_recall(expected_keywords, contexts)

    evaluation_results.append({
        'query_id': row['query_id'],
        'question': question,
        'generated_answer': answer,
        'answer_relevance': answer_relevance,
        'groundedness': groundedness,
        'context_precision': context_precision,
        'context_recall': context_recall
    })

ragas_style_results_df = pd.DataFrame(evaluation_results)
ragas_style_results_df.to_csv(OUTPUT_PATH / 'ragas_style_evaluation_results.csv', index=False)

ragas_style_results_df

,query_id,question,generated_answer,answer_relevance,groundedness,context_precision,context_recall
0,Q1,What quality issue was found for equipment EQP...,"Based on the retrieved enterprise context, the...",0.7273,0.5645,0.8333,0.8333
1,Q2,Which supplier invoice mentions material issue...,"Based on the retrieved enterprise context, the...",0.5556,0.5714,0.6667,0.6667
2,Q3,What maintenance record mentions downtime impa...,"Based on the retrieved enterprise context, the...",0.6667,0.5574,0.7500,0.7500
3,Q4,Which engineering change notice explains downs...,"Based on the retrieved enterprise context, the...",0.8750,0.5397,1.0000,1.0000


In [15]:
all_retrieval_rows = []

for _, row in evaluation_df.iterrows():
    question = row['question']
    query_id = row['query_id']
    retrieved_df = retrieve_top_k_chunks(question, top_k=5)

    for rank_idx, (_, r) in enumerate(retrieved_df.iterrows(), start=1):
        all_retrieval_rows.append({
            'query_id': query_id,
            'question': question,
            'rank': rank_idx,
            'chunk_id': r['chunk_id'],
            'document_id': r['document_id'],
            'document_type': r['document_type'],
            'plant_code': r['plant_code'],
            'supplier_name': r['supplier_name'],
            'equipment_id': r['equipment_id'],
            'similarity_score': r['similarity_score'],
            'chunk_text': r['chunk_text']
        })

retrieval_details_df = pd.DataFrame(all_retrieval_rows)
retrieval_details_df.to_csv(OUTPUT_PATH / 'retrieval_details.csv', index=False)

retrieval_details_df.head(10)

,query_id,question,rank,chunk_id,document_id,document_type,plant_code,supplier_name,equipment_id,similarity_score,chunk_text
0,Q1,What quality issue was found for equipment EQP...,1,DOC00298_CHUNK_1,DOC00298,quality_report,PLT200,Aptiv,EQP1002,0.400794,document id doc00298. plant plt200. quality in...
1,Q1,What quality issue was found for equipment EQP...,2,DOC00360_CHUNK_1,DOC00360,quality_report,PLT200,Bosch,EQP1002,0.400098,document id doc00360. plant plt200. quality in...
2,Q1,What quality issue was found for equipment EQP...,3,DOC00107_CHUNK_1,DOC00107,quality_report,PLT200,Denso,EQP1002,0.398952,document id doc00107. plant plt200. quality in...
3,Q1,What quality issue was found for equipment EQP...,4,DOC00041_CHUNK_1,DOC00041,quality_report,PLT200,Aptiv,EQP1002,0.398884,document id doc00041. plant plt200. quality in...
4,Q1,What quality issue was found for equipment EQP...,5,DOC00364_CHUNK_1,DOC00364,quality_report,PLT200,Denso,EQP1002,0.398559,document id doc00364. plant plt200. quality in...
5,Q2,Which supplier invoice mentions material issue...,1,DOC00011_CHUNK_1,DOC00011,supplier_invoice,PLT100,Denso,None,0.387760,document id doc00011. supplier invoice from de...
6,Q2,Which supplier invoice mentions material issue...,2,DOC00173_CHUNK_1,DOC00173,supplier_invoice,PLT400,Lear,None,0.383484,document id doc00173. supplier invoice from le...
7,Q2,Which supplier invoice mentions material issue...,3,DOC00125_CHUNK_1,DOC00125,supplier_invoice,PLT400,Bosch,None,0.381471,document id doc00125. supplier invoice from bo...
8,Q2,Which supplier invoice mentions material issue...,4,DOC00308_CHUNK_1,DOC00308,supplier_invoice,PLT300,Bosch,None,0.380689,document id doc00308. supplier invoice from bo...
9,Q2,Which supplier invoice mentions material issue...,5,DOC00197_CHUNK_1,DOC00197,supplier_invoice,PLT300,Denso,None,0.380135,document id doc00197. supplier invoice from de...


In [16]:
benchmark_summary = {
    'generated_at': datetime.now().isoformat(),
    'question_count': int(len(ragas_style_results_df)),
    'average_answer_relevance': round(float(ragas_style_results_df['answer_relevance'].mean()), 4),
    'average_groundedness': round(float(ragas_style_results_df['groundedness'].mean()), 4),
    'average_context_precision': round(float(ragas_style_results_df['context_precision'].mean()), 4),
    'average_context_recall': round(float(ragas_style_results_df['context_recall'].mean()), 4)
}

with open(OUTPUT_PATH / 'benchmark_summary.json', 'w') as f:
    json.dump(benchmark_summary, f, indent=2)

print(json.dumps(benchmark_summary, indent=2))

{
  "generated_at": "2026-03-17T22:16:21.027110",
  "question_count": 4,
  "average_answer_relevance": 0.7061,
  "average_groundedness": 0.5582,
  "average_context_precision": 0.8125,
  "average_context_recall": 0.8125
}


In [17]:
print('--- RAGAS-STYLE EVALUATION RESULTS ---')
display(ragas_style_results_df)

print('\n--- RETRIEVAL DETAILS ---')
display(retrieval_details_df.head(10))

print('\n--- BENCHMARK SUMMARY ---')
print(json.dumps(benchmark_summary, indent=2))

--- RAGAS-STYLE EVALUATION RESULTS ---


,query_id,question,generated_answer,answer_relevance,groundedness,context_precision,context_recall
0,Q1,What quality issue was found for equipment EQP...,"Based on the retrieved enterprise context, the...",0.7273,0.5645,0.8333,0.8333
1,Q2,Which supplier invoice mentions material issue...,"Based on the retrieved enterprise context, the...",0.5556,0.5714,0.6667,0.6667
2,Q3,What maintenance record mentions downtime impa...,"Based on the retrieved enterprise context, the...",0.6667,0.5574,0.7500,0.7500
3,Q4,Which engineering change notice explains downs...,"Based on the retrieved enterprise context, the...",0.8750,0.5397,1.0000,1.0000



--- RETRIEVAL DETAILS ---


,query_id,question,rank,chunk_id,document_id,document_type,plant_code,supplier_name,equipment_id,similarity_score,chunk_text
0,Q1,What quality issue was found for equipment EQP...,1,DOC00298_CHUNK_1,DOC00298,quality_report,PLT200,Aptiv,EQP1002,0.400794,document id doc00298. plant plt200. quality in...
1,Q1,What quality issue was found for equipment EQP...,2,DOC00360_CHUNK_1,DOC00360,quality_report,PLT200,Bosch,EQP1002,0.400098,document id doc00360. plant plt200. quality in...
2,Q1,What quality issue was found for equipment EQP...,3,DOC00107_CHUNK_1,DOC00107,quality_report,PLT200,Denso,EQP1002,0.398952,document id doc00107. plant plt200. quality in...
3,Q1,What quality issue was found for equipment EQP...,4,DOC00041_CHUNK_1,DOC00041,quality_report,PLT200,Aptiv,EQP1002,0.398884,document id doc00041. plant plt200. quality in...
4,Q1,What quality issue was found for equipment EQP...,5,DOC00364_CHUNK_1,DOC00364,quality_report,PLT200,Denso,EQP1002,0.398559,document id doc00364. plant plt200. quality in...
5,Q2,Which supplier invoice mentions material issue...,1,DOC00011_CHUNK_1,DOC00011,supplier_invoice,PLT100,Denso,None,0.387760,document id doc00011. supplier invoice from de...
6,Q2,Which supplier invoice mentions material issue...,2,DOC00173_CHUNK_1,DOC00173,supplier_invoice,PLT400,Lear,None,0.383484,document id doc00173. supplier invoice from le...
7,Q2,Which supplier invoice mentions material issue...,3,DOC00125_CHUNK_1,DOC00125,supplier_invoice,PLT400,Bosch,None,0.381471,document id doc00125. supplier invoice from bo...
8,Q2,Which supplier invoice mentions material issue...,4,DOC00308_CHUNK_1,DOC00308,supplier_invoice,PLT300,Bosch,None,0.380689,document id doc00308. supplier invoice from bo...
9,Q2,Which supplier invoice mentions material issue...,5,DOC00197_CHUNK_1,DOC00197,supplier_invoice,PLT300,Denso,None,0.380135,document id doc00197. supplier invoice from de...



--- BENCHMARK SUMMARY ---
{
  "generated_at": "2026-03-17T22:16:21.027110",
  "question_count": 4,
  "average_answer_relevance": 0.7061,
  "average_groundedness": 0.5582,
  "average_context_precision": 0.8125,
  "average_context_recall": 0.8125
}
